# Tolerance Range Selection
## Context
To calculate the travel time between each stop for every bus trip, we perform a complex asof_join, which requires a manual setting of `tolerance` range to work well.
## Goal
Find the ideal tolerance for each stop pair for each route.

## (below for future documentation)
## Find ideal tolerance in asof_join
1. Required data: `bus_event_time.parquet`, `target_stop.json`
2. Create ideal_tolerances = []
2. For each route, set tolerance = '1h'
3. Perform asof_join, get the number of rows of resulting joined df (`nrows: int`)
3. Repeat the following
    - increase tolerance by '1h' (new_tolerance)
    - perform asof_join again, get new_nrows
    - if new_nrows / nrows < 1.03 or tolerance == "12h", ideal_tolerance = tolerance, break the loop
    - else, tolerance = new_tolerance, continue the loop
6. ideal_tolerances.append({"depart": int, "dest": int, "tolerance": str})
7. Create pl.DataFrame(ideal_tolerances)
8. df.write_csv

# Import

In [1]:
import polars as pl
from polars import Series
from constants import DATA_FILE, PROCESSED_DATA_FOLDER
from helpers import clean_df
import json
import plotly.express as px
from datetime import datetime, date, time, timedelta

In [2]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes = json.load(f)
df_train = (
    pl.scan_parquet(DATA_FILE)
    .filter(pl.col("SubRouteID").is_in(target_routes))
    .pipe(clean_df)
)
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r", encoding="utf-8") as f:
    target_stops = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict = json.load(f)

In [3]:
df_train.select(pl.col("StopID").unique()).collect().describe()

statistic,StopID
str,f64
"""count""",19156.0
"""null_count""",0.0
"""mean""",234711.469409
"""std""",66470.785466
"""min""",21282.0
"""25%""",225726.0
"""50%""",256302.0
"""75%""",278215.0
"""max""",312015.0


In [4]:
def asof_join_by_id(
    route_df: pl.DataFrame,
    depart_id: int,
    dest_id: int,
    tolerance: str,
) -> int:
    depart_df = (
        route_df.filter(
            pl.col("Event") == "離站",
            pl.col("StopID") == depart_id,
        )
        .with_columns(pl.col("Time").alias("DepartureTime"))
        .sort("Time")
    )
    dest_df = (
        route_df.filter(
            pl.col("Event") == "進站",
            pl.col("StopID") == dest_id,
        )
        .with_columns(pl.col("Time").alias("ArrivalTime"))
        .sort("Time")
    )
    result = depart_df.join_asof(
        dest_df,
        on="Time",
        by="Plate",
        strategy="forward",
        tolerance=tolerance,
        check_sortedness=False,
    ).drop_nulls(subset=["ArrivalTime"])
    return len(result)

In [5]:
def find_ideal_tolerances(
    df: pl.LazyFrame,
    target_stops: dict[str, list[int]],
) -> pl.DataFrame:
    """
    Finds the ideal asof_join tolerance for each sequential stop pair per route.

    Args:
        df: Cleaned LazyFrame (output of clean_df).
        target_stops: Maps Route (str) to an ordered list of StopIDs (int).
                      Pairs are derived from consecutive stops in the list.

    Returns:
        A DataFrame with columns: route, depart_id, dest_id, tolerance.
    """

    def _find_tolerance_for_pair(
        route_df: pl.DataFrame,
        depart_id: int,
        dest_id: int,
    ) -> tuple[str, int]:
        MAX_HOURS = 12
        tolerance_hours = 1
        n_rows = asof_join_by_id(route_df, depart_id, dest_id, f"{tolerance_hours}h")

        while tolerance_hours < MAX_HOURS:
            new_tolerance_hours = tolerance_hours + 1
            new_nrows = asof_join_by_id(
                route_df, depart_id, dest_id, f"{new_tolerance_hours}h"
            )
            if n_rows > 0 and new_nrows / n_rows < 1.03:
                break
            tolerance_hours = new_tolerance_hours
            n_rows = new_nrows

        return (f"{tolerance_hours}h", n_rows)

    collected_df = df.select(["Route", "Plate", "StopID", "Event", "Time"]).collect(
        engine="streaming"
    )
    results: list[dict] = []
    count = 0
    for route, stop_ids in target_stops.items():
        route_df = collected_df.filter(pl.col("Route") == route)
        if route_df.is_empty():
            continue

        for depart_id, dest_id in zip(stop_ids, stop_ids[1:]):
            tolerance, n_rows = _find_tolerance_for_pair(route_df, depart_id, dest_id)
            results.append(
                {
                    "route": route,
                    "depart_id": depart_id,
                    "dest_id": dest_id,
                    "tolerance": tolerance,
                    "n_rows": n_rows,
                }
            )
        count += 1
        if count % 100 == 0:
            print(f"{count} routes have been processed.")

    return pl.DataFrame(
        results,
        schema={
            "route": pl.String,
            "depart_id": pl.Int32,
            "dest_id": pl.Int32,
            "tolerance": pl.String,
            "n_rows": pl.UInt16,
        },
    )

In [6]:
ideal_tolerances = find_ideal_tolerances(df_train, target_stops)

100 routes have been processed.
200 routes have been processed.
300 routes have been processed.
400 routes have been processed.
500 routes have been processed.
600 routes have been processed.
700 routes have been processed.
800 routes have been processed.
900 routes have been processed.
1000 routes have been processed.
1100 routes have been processed.
1200 routes have been processed.
1300 routes have been processed.
1400 routes have been processed.
1500 routes have been processed.
1600 routes have been processed.


- It took 9 minutes 49 seconds to finish 1666 routes.

In [8]:
ideal_tolerances.write_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv")

In [53]:
hour_order = ["1h", "2h", "3h", "4h", "5h", "6h", "12h"]

fig = px.histogram(
    ideal_tolerances.sort("tolerance", descending=True),
    x="tolerance",
    category_orders={"tolerance": hour_order},
    color="tolerance",
    color_discrete_sequence=px.colors.sequential.Teal,
    labels={"tolerance": "Tolerance window", "count": "Count"},
    title="Distribution of ideal tolerances",
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter, sans-serif", size=13, color="#444"),
    title=dict(
        font=dict(size=16, color="#222"), x=0, xanchor="left", pad=dict(l=0, b=16)
    ),
    xaxis=dict(
        categoryorder="array",
        categoryarray=hour_order,
        showgrid=False,
        showline=True,
        linecolor="#ddd",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        showline=False,
        tickfont=dict(size=12),
    ),
    bargap=0.25,
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(marker_line_width=0)

fig.show()

In [16]:
fig = px.histogram(
    ideal_tolerances,
    x="n_rows",
    labels={"n_rows": "Number of Rows", "count": "Count"},
    color_discrete_sequence=["#1FC2B0"],
    title="Distribution of row counts",
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter, sans-serif", size=13, color="#444"),
    title=dict(
        font=dict(size=16, color="#222"), x=0, xanchor="left", pad=dict(l=0, b=16)
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor="#ddd",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        showline=False,
        tickfont=dict(size=12),
    ),
    bargap=0.25,
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(marker_line_width=0)

fig.show()

In [79]:
ideal_tolerances.with_columns(
    pl.col("depart_id")
    .replace_strict(
        old=Series(stops_dict.keys()), new=Series(stops_dict.values()), default="None"
    )
    .alias("Depart_name"),
    pl.col("dest_id")
    .replace_strict(
        old=Series(stops_dict.keys()), new=Series(stops_dict.values()), default="None"
    )
    .alias("Dest_name"),
).drop(["depart_id", "dest_id"]).filter(pl.col("route") == "560802")


route,tolerance,Depart_name,Dest_name
str,str,str,str
"""560802""","""1h""","""下公館站""","""托盤山"""
"""560802""","""1h""","""托盤山""","""科學園區"""
"""560802""","""1h""","""科學園區""","""中正路"""


In [66]:
Series(stops_dict.values())

""
str
"""仁愛新生路口"""
"""光華商場"""
"""光華商場"""
"""臺北車站(鄭州)"""
"""酒泉重慶路口"""
…
"""東竹"""
"""東竹"""
"""馬加錄"""


In [52]:
ideal_tolerances.filter(pl.col("tolerance") == "12h")

route,depart_id,dest_id,tolerance
str,i64,i64,str
"""183701""",268417,311648,"""12h"""
"""183802""",268481,268417,"""12h"""
"""750002""",303140,294043,"""12h"""
"""750002""",294043,189248,"""12h"""
"""1613A2""",266866,300440,"""12h"""
…,…,…,…
"""186201""",268417,268482,"""12h"""
"""186202""",268481,268417,"""12h"""
"""1633C2""",266216,300702,"""12h"""


In [27]:
test = df_train.filter(
    pl.col("Route") == "183701", pl.col("StopID").is_in([268417, 311648])
).collect()

In [31]:
for i in range(12):
    print(f"{i + 1}h:")
    print(asof_join_by_id(test, 268417, 311648, f"{i + 1}h"))

1h:
0
2h:
0
3h:
0
4h:
0
5h:
0
6h:
0
7h:
0
8h:
0
9h:
0
10h:
0
11h:
0
12h:
0


In [50]:
import time as t

for days in range(30):
    d = date(2025, 10, 21) + timedelta(days=days)
    display(
        test.filter(
            pl.col("Time").is_between(d, datetime.combine(d, time(23, 59, 59))),
        )
    )
    t.sleep(5)

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-2627""","""去程""","""永康轉運站""",311648,4,"""進站""",2025-10-21 01:49:41,"""Tue"""
"""183701""","""KKA-2627""","""去程""","""永康轉運站""",311648,4,"""離站""",2025-10-21 01:51:08,"""Tue"""
"""183701""","""KKA-1182""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-21 10:07:08,"""Tue"""
"""183701""","""KKA-1182""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-21 10:09:39,"""Tue"""
"""183701""","""KKA-1173""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-21 18:36:44,"""Tue"""
"""183701""","""KKA-1173""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-21 18:40:35,"""Tue"""


Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-2622""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-22 10:15:40,"""Wed"""
"""183701""","""KKA-2622""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-22 10:18:37,"""Wed"""
"""183701""","""KKA-9806""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-22 18:34:45,"""Wed"""
"""183701""","""KKA-9806""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-22 18:36:23,"""Wed"""


Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-23 10:31:39,"""Thu"""
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-23 10:31:47,"""Thu"""
"""183701""","""KKA-1171""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-23 19:05:39,"""Thu"""
"""183701""","""KKA-1171""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-23 19:09:23,"""Thu"""
"""183701""","""KKA-1180""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-23 21:07:41,"""Thu"""
"""183701""","""KKA-1180""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-23 21:11:26,"""Thu"""


Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-2680""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-24 01:02:54,"""Fri"""
"""183701""","""KKA-2680""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-24 01:07:33,"""Fri"""
"""183701""","""KKA-1183""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-24 02:14:43,"""Fri"""
"""183701""","""KKA-1183""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-24 02:17:41,"""Fri"""
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-24 10:55:47,"""Fri"""
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-24 10:59:37,"""Fri"""
"""183701""","""KKA-2607""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-24 19:13:46,"""Fri"""
"""183701""","""KKA-2607""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-24 19:17:15,"""Fri"""
"""183701""","""KKA-1179""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-24 23:49:37,"""Fri"""


Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-25 10:19:43,"""Sat"""
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-25 10:19:52,"""Sat"""
"""183701""","""KKA-9807""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-25 14:39:46,"""Sat"""
"""183701""","""KKA-9807""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-25 14:42:16,"""Sat"""


Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-26 10:11:38,"""Sun"""
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-26 10:16:24,"""Sun"""
"""183701""","""KKA-2608""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-26 13:57:57,"""Sun"""
"""183701""","""KKA-2608""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-26 14:00:45,"""Sun"""


KeyboardInterrupt: 

In [41]:
test.filter(pl.col("StopID") == 311648)

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-2627""","""去程""","""永康轉運站""",311648,4,"""進站""",2025-10-21 01:49:41,"""Tue"""
"""183701""","""KKA-2627""","""去程""","""永康轉運站""",311648,4,"""離站""",2025-10-21 01:51:08,"""Tue"""
"""183701""","""KKA-1182""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-21 10:07:08,"""Tue"""
"""183701""","""KKA-1182""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-10-21 10:09:39,"""Tue"""
"""183701""","""KKA-1173""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-10-21 18:36:44,"""Tue"""
…,…,…,…,…,…,…,…,…
"""183701""","""KKA-1209""","""去程""","""永康轉運站""",311648,5,"""離站""",2026-02-27 11:22:34,"""Fri"""
"""183701""","""KKA-1167""","""去程""","""永康轉運站""",311648,5,"""進站""",2026-02-27 15:21:42,"""Fri"""
"""183701""","""KKA-1167""","""去程""","""永康轉運站""",311648,5,"""離站""",2026-02-27 15:24:35,"""Fri"""
